# NaturalSym — solved inputs that look like real data

You know the shape of your own data; a solver does not, and left to itself satisfies
`amount > 100` with `101` every time. Declare the shape and it will not.

In [ ]:
import os, sys, glob

ROOT = os.environ.get("BIGASTERISK_HOME") or os.path.abspath("..")

# Jars: a source checkout has them under modules/*/target, the Docker image under jars/.
JARS = sorted(glob.glob(f"{ROOT}/modules/*/target/scala-2.13/bigasterisk-*.jar")) \
    or sorted(glob.glob(f"{ROOT}/jars/bigasterisk-*.jar"))
if not JARS:
    raise SystemExit("No BigAsterisk jars found. Run: bin/sbt package")

FASTUTIL_JAR = os.environ.get("FASTUTIL_JAR") or next(iter(sorted(
    glob.glob(f"{ROOT}/jars/fastutil*.jar")
    + glob.glob(os.path.expanduser("~/Library/Caches/Coursier/**/fastutil-8.5.15.jar"), recursive=True)
    + glob.glob(os.path.expanduser("~/.cache/coursier/**/fastutil-8.5.15.jar"), recursive=True)
)), None)
if not FASTUTIL_JAR:
    raise SystemExit("fastutil jar not found. Run: bin/sbt package")

SPARK_JARS = ",".join(JARS + [FASTUTIL_JAR])
DATA = f"{ROOT}/examples/data"
sys.path.insert(0, f"{ROOT}/python")

## The data

Twelve orders across three customers. One of them, `o8`, is an outlier at
`99999` — every notebook here uses it as the thing to find.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import bigasterisk

spark = (bigasterisk.configure(SparkSession.builder)
    .master("local[2]")
    .appName("naturalsym-notebook")
    .config("spark.jars", SPARK_JARS)
    .config("spark.sql.adaptive.skewJoin.enabled", "false")
    .config("spark.ui.enabled", "false")
    .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

orders = spark.read.schema("oid STRING, cid STRING, amount INT").csv(f"{DATA}/orders.txt")
customers = spark.read.schema("cid STRING, name STRING").csv(f"{DATA}/customers.txt")
orders.createOrReplaceTempView("orders")
customers.createOrReplaceTempView("customers")

orders.show()

## Declare the shape of each column

In [ ]:
suite = bigasterisk.testgen(spark).generate(
    "SELECT cid FROM orders WHERE amount > 100",
    {"orders": orders}, rows_per_path=5,
    distributions={"cid": 'Discrete("north", "south")',
                   "amount": "uniform(101, 400)"})

for case in suite.cases:
    print(case)

## Naturalness never costs coverage

`binom(100, 0.1)` can never exceed 100, so it cannot satisfy the branch. The solver falls back to a value that can.

In [ ]:
fallback = bigasterisk.testgen(spark).generate(
    "SELECT cid FROM orders WHERE amount > 100",
    {"orders": orders}, rows_per_path=1,
    distributions={"amount": "binom(100, 0.1)"})

taking = [c for c in fallback.cases if not c.path.startswith("NOT")][0]
print(taking)

## Check

In [ ]:
declared = [c for c in suite.cases if not c.path.startswith("NOT")][0]
assert all("north" in r or "south" in r for r in declared.tables["orders"]), \
    declared.tables
assert taking.verified, taking.note
print("OK")